In [2]:
import pandas as pd
from pathlib import Path
import yaml
import sys
import yaml

In [3]:
path_yaml = Path("../src/config.yaml")
try:

    with open(path_yaml, "r") as file:
        config = yaml.safe_load(file)

except FileNotFoundError:
    print("Config file not found")
    sys.exit(1)


In [4]:
mapping = config["mapping"]
real_mapping = {}

for k, v in mapping.items():
    real_mapping[v] = k

In [5]:
path_raw = Path(config["paths"]["raw"])
path_processed = Path(config["paths"]["processed"])
useless_columns = config["features"]["useless_columns"]
raw_standardization = config["data_standardization"]
extra_services_columns = config["features"]["extra_services"]

df = pd.read_csv(path_raw)

pd.set_option('display.max_columns', None)
pd.set_option('display.max_colwidth', None)
pd.set_option('future.no_silent_downcasting', True)

In [6]:
df = df[df["Churn Value"] == 0]

In [7]:
df = df.drop(columns=useless_columns)

In [8]:
df = df.rename(columns=real_mapping)

In [9]:
map_to_apply = {}
for std_key, raw_values in config["data_standardization"].items():
    if isinstance(raw_values, list):
        for val in raw_values:
            map_to_apply[val] = std_key
    else:
        map_to_apply[raw_values] = std_key

df = df.replace(map_to_apply)

In [11]:
for col in config["features"]["nominal_categoricals"]:
    df[col] = df[col].astype(str).str.lower().str.strip()

In [10]:
df["Total Extra Services"] = df[extra_services_columns].eq("yes").sum(axis = 1)

In [11]:
df["is_fiber"] = (df["internet_service"] == "fiber_optic").astype(int)
df = df.drop(columns=["internet_service"])

In [12]:
df.set_index("customer_id", inplace=True)

In [13]:
df

,gender,senior_citizen,partner,dependents,tenure_months,phone_service,multiple_lines,online_security,online_backup,device_protection,tech_support,streaming_tv,streaming_movies,contract_type,paperless_billing,payment_method,cltv,Total Extra Services,is_fiber
customer_id,,,,,,,,,,,,,,,,,,,
7590-VHVEG,female,no,yes,no,1,no,no,no,yes,no,no,no,no,month_to_month,yes,electronic_check,3964,1,0
5575-GNVDE,male,no,no,no,34,yes,no,yes,no,yes,no,no,no,one_year,no,mailed_check,3441,3,0
7795-CFOCW,male,no,no,no,45,no,no,yes,no,yes,yes,no,no,one_year,no,bank_transfer,4307,2,0
1452-KIOVK,male,no,no,yes,22,yes,yes,no,yes,no,no,yes,no,month_to_month,yes,credit_card,4459,4,1
6713-OKOMC,female,no,no,no,10,no,no,yes,no,no,no,no,no,month_to_month,no,mailed_check,2013,1,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
2569-WGERO,female,no,no,no,72,yes,no,no,no,no,no,no,no,two_year,yes,bank_transfer,5306,1,0
6840-RESVB,male,no,yes,yes,24,yes,yes,yes,no,yes,yes,yes,yes,one_year,yes,mailed_check,2140,6,0
2234-XADUH,female,no,yes,yes,72,yes,yes,no,yes,yes,no,yes,yes,one_year,yes,credit_card,5560,6,1


In [14]:
df.to_csv(path_processed.joinpath("telco_preprocessed.csv"), index=True)